## Training scheme for https://link.springer.com/content/pdf/10.1007/978-3-031-93688-3_19.pdf
## But using a GCN

- Building training data for detector

In [ ]:
# Frame-level punch / no-punch labels from Excel (V1–V10), aligned to source RGB MP4s.
# 0 = no punch, 1 = inside some annotated punch interval (inclusive 1-based [start,end]).
# Gaps between clips (e.g. frames 5–6 between [1–4] and [7–9]) stay 0 — full timeline covered.
# Also saves sliding-window rows for GCNDetector-style training (window_length=T, label=1 if any punch frame in window).

from pathlib import Path

import cv2
import numpy as np

from preprocess import _find_video, _load_annotations

_REPO = Path.cwd().resolve()
assert (_REPO / "preprocess.py").exists(), "Run this notebook with working directory = repo root (pose/)."

_ANNOT = _REPO / "Dataset" / "Annotation_files"
_OUT = _REPO / "Dataset" / "detection_frame_labels"
_OUT.mkdir(parents=True, exist_ok=True)

WINDOW_LENGTH = 11  # temporal length for GCNDetector / Baghel-style windows
STRIDE = 1
# Use only the first N frames of each MP4 in this npz (None = full video).
MAX_VIDEO_FRAMES = 256


def _video_frame_count(path: Path) -> int:
    cap = cv2.VideoCapture(str(path))
    if not cap.isOpened():
        return 0
    n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()
    return max(0, n)


def _frame_punch_binary(num_frames: int, annotations: list[tuple[int, int, str]]) -> np.ndarray:
    """One label per video frame. Annotations use the same 1-based inclusive convention as preprocess.extract_landmarks."""
    y = np.zeros(num_frames, dtype=np.uint8)
    for s, e, _ in annotations:
        if e < s:
            continue
        s0 = max(0, s - 1)
        e_excl = min(num_frames, e)  # exclusive end index: 1-based inclusive e → indices … e-1
        if s0 < e_excl:
            y[s0:e_excl] = 1
    return y


def _sliding_windows(frame_y: np.ndarray, window: int, stride: int) -> tuple[np.ndarray, np.ndarray]:
    """window_is_punch[k] = 1 iff max(frame_y[t:t+window]) == 1 (any punch frame in the window)."""
    if len(frame_y) < window:
        return np.zeros(0, dtype=np.int64), np.zeros(0, dtype=np.uint8)
    starts = np.arange(0, len(frame_y) - window + 1, stride, dtype=np.int64)
    win_y = np.array(
        [int(frame_y[t : t + window].max() > 0) for t in starts],
        dtype=np.uint8,
    )
    return starts, win_y


for i in range(1, 11):
    ver = f"V{i}"
    vid = _find_video(ver)
    xlsx = _ANNOT / f"{ver}.xlsx"
    if vid is None or not xlsx.exists():
        print(f"[skip] {ver}: missing video or {xlsx.name}")
        continue

    ann = _load_annotations(xlsx)
    F_full = _video_frame_count(vid)
    if F_full == 0:
        print(f"[skip] {ver}: zero frames")
        continue

    frame_y = _frame_punch_binary(F_full, ann)
    if MAX_VIDEO_FRAMES is not None:
        F = min(F_full, int(MAX_VIDEO_FRAMES))
        frame_y = frame_y[:F]
    else:
        F = F_full

    w_starts, w_y = _sliding_windows(frame_y, WINDOW_LENGTH, STRIDE)

    cap = cv2.VideoCapture(str(vid))
    fps = float(cap.get(cv2.CAP_PROP_FPS) or 0.0)
    cap.release()

    intervals = (
        np.array([[s, e] for s, e, _ in ann], dtype=np.int32)
        if ann
        else np.zeros((0, 2), dtype=np.int32)
    )

    out = _OUT / f"{ver}_detection.npz"
    np.savez_compressed(
        out,
        punch_frame_binary=frame_y,
        num_frames=np.int32(F),
        fps=np.float32(fps),
        window_length=np.int32(WINDOW_LENGTH),
        stride=np.int32(STRIDE),
        window_starts=w_starts,
        window_is_punch=w_y,
        punch_intervals_1based=intervals,
        version=np.array(ver),
        source_video=np.array(str(vid)),
    )

    print(
        f"{ver}: frames={F}  punch_frames={int(frame_y.sum())}  "
        f"windows={len(w_y)}  punch_windows={int(w_y.sum())}  -> {out.relative_to(_REPO)}"
    )

print(f"\nDone. Labels under {_OUT.relative_to(_REPO)}/")
print("Pair each window start index with pose windows [N,M,T,V,C] when you stack skeleton clips.")

### Outputs (`Dataset/detection_frame_labels/`)

For each `V1` … `V10`, **`{ver}_detection.npz`** contains:

| Key | Meaning |
|-----|--------|
| `punch_frame_binary` | `(num_frames,)` uint8 — 0/1 for every frame of the MP4 |
| `num_frames`, `fps` | Video length and FPS |
| `punch_intervals_1based` | `(N, 2)` Excel intervals `[start, end]` (inclusive, 1-based) |
| `window_length`, `stride` | Defaults 11 and 1 |
| `window_starts` | Start indices `t` for windows `[t : t+window_length)` |
| `window_is_punch` | 1 if **any** frame in that window is punch (for detector targets) |
| `version`, `source_video` | Strings |

**GCNDetector** expects clips `[N, M, T, V, C]` with `T = window_length`.

## Punch detector training (`GCNDetector`)

1. **Build window cache** (next cell): one MediaPipe pass over the RGB video, then stack 11-frame clips aligned with `window_starts` / `window_is_punch`. Use `max_windows` to cap size (balanced punch vs no-punch). *Slow* on long videos.
2. **Train** (following cell): `GCNDetector` (BoxingVI 12 joints), `BCELoss`, AdamW.

Restart the kernel after editing `GCN.py` or `detector_data.py`.

In [ ]:
# --- Build pose cache for one workbook (repeat or loop V1–V10) ---
from pathlib import Path

from preprocess import _find_video
from detector_data import build_detector_windows_npz

_REPO = Path.cwd().resolve()
DET_VER = "V1"

DET_NPZ = _REPO / "Dataset" / "detection_frame_labels" / f"{DET_VER}_detection.npz"
OUT_CACHE = _REPO / "Dataset" / "detector_training" / f"{DET_VER}_windows.npz"
VIDEO = _find_video(DET_VER)

MAX_WINDOWS = 8000  # None = keep every window after pose extract (very large)

assert DET_NPZ.exists(), "Run the preprocessing cell first."
assert VIDEO is not None and VIDEO.exists(), f"No MP4 for {DET_VER}"

build_detector_windows_npz(
    DET_NPZ,
    VIDEO,
    OUT_CACHE,
    max_windows=MAX_WINDOWS,
    balance=True,
    seed=42,
)
print(f"Saved detector cache → {OUT_CACHE.relative_to(_REPO)}")

I0000 00:00:1778171825.047182   38029 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1778171825.086822   38047 gl_context.cc:385] GL version: 3.2 (OpenGL ES 3.2 NVIDIA 580.95.05), renderer: NVIDIA GeForce RTX 5070/PCIe/SSE2
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1778171825.112845   38033 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1778171825.126073   38037 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1778171825.151760   38038 landmark_projection_calculator.cc:78] Using NORM_RECT without IMAGE_DIMENSIONS is only supported for the square ROI. Provide IMAGE_DIMENSIONS or use PROJECTION_MATRIX.


Frame:  0 / 46559
Frame:  100 / 46559
Frame:  200 / 46559
Frame:  300 / 46559
Frame:  400 / 46559
Frame:  500 / 46559
Frame:  600 / 46559
Frame:  700 / 46559
Frame:  800 / 46559
Frame:  900 / 46559
Frame:  1000 / 46559
Frame:  1100 / 46559
Frame:  1200 / 46559
Frame:  1300 / 46559
Frame:  1400 / 46559
Frame:  1500 / 46559
Frame:  1600 / 46559
Frame:  1700 / 46559
Frame:  1800 / 46559
Frame:  1900 / 46559
Frame:  2000 / 46559
Frame:  2100 / 46559
Frame:  2200 / 46559
Frame:  2300 / 46559
Frame:  2400 / 46559
Frame:  2500 / 46559
Frame:  2600 / 46559
Frame:  2700 / 46559
Frame:  2800 / 46559
Frame:  2900 / 46559
Frame:  3000 / 46559
Frame:  3100 / 46559
Frame:  3200 / 46559
Frame:  3300 / 46559
Frame:  3400 / 46559
Frame:  3500 / 46559
Frame:  3600 / 46559
Frame:  3700 / 46559
Frame:  3800 / 46559
Frame:  3900 / 46559
Frame:  4000 / 46559
Frame:  4100 / 46559
Frame:  4200 / 46559
Frame:  4300 / 46559
Frame:  4400 / 46559
Frame:  4500 / 46559
Frame:  4600 / 46559
Frame:  4700 / 46559
Fram

In [6]:
# --- Train GCNDetector on cached windows ---
import torch
from pathlib import Path

from torch.utils.data import ConcatDataset, DataLoader, random_split

from GCN import (
    BOXINGVI_BONE_PAIRS,
    BOXINGVI_CENTER_JOINT,
    BOXINGVI_GRAPH_EDGES,
    GCNDetector,
    NUM_BOXINGVI_JOINTS,
)
from detector_data import DetectorWindowNpzDataset

_REPO = Path.cwd().resolve()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Build one `{ver}_windows.npz` per entry (detector cache cell), then list them here.
DATA_VERS = ["V7"]
_CACHEDIR = _REPO / "Dataset" / "detector_training"
_CACHES = [_CACHEDIR / f"{v}_windows.npz" for v in DATA_VERS]
for p in _CACHES:
    assert p.exists(), f"Missing cache — build it first: {p}"

# Overfitting sanity check: strip regularization + train longer → train acc should approach 1.0.
# Set False when tuning for validation/generalization again.
OVERFIT_SANITY = True

if OVERFIT_SANITY:
    EPOCHS = 80
    LR = 3e-3
    WEIGHT_DECAY = 0.0
    BACKBONE_DROPOUT = 0.0
else:
    EPOCHS = 15
    LR = 1e-3
    WEIGHT_DECAY = 1e-4
    BACKBONE_DROPOUT = 0.1

BATCH_SIZE = 48
VAL_FRAC = 0.15
SEED = 42

torch.manual_seed(SEED)
_parts = [DetectorWindowNpzDataset(p) for p in _CACHES]
full_ds = ConcatDataset(_parts)
print(f"ConcatDataset: {len(DATA_VERS)} files, {len(full_ds)} windows total")
for v, ds in zip(DATA_VERS, _parts):
    print(f"  {v}: {len(ds)}")
if OVERFIT_SANITY:
    print(
        "OVERFIT_SANITY: weight_decay=0, backbone dropout=0 — expect train acc → ~1.0 "
        "(val may degrade)."
    )

n_val = max(1, int(round(VAL_FRAC * len(full_ds))))
n_train = len(full_ds) - n_val
train_ds, val_ds = random_split(
    full_ds,
    [n_train, n_val],
    generator=torch.Generator().manual_seed(SEED),
)

train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=False, num_workers=0)
val_dl = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

model = GCNDetector(
    in_channels=2,
    num_joints=NUM_BOXINGVI_JOINTS,
    bone_pairs=BOXINGVI_BONE_PAIRS,
    backbone_kwargs={
        "edges": BOXINGVI_GRAPH_EDGES,
        "center": BOXINGVI_CENTER_JOINT,
        "dropout": BACKBONE_DROPOUT,
        "data_bn": True,
    },
    dropout=0,
).to(DEVICE)

opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
loss_fn = torch.nn.BCELoss()


@torch.no_grad()
def evaluate(loader: DataLoader) -> tuple[float, float]:
    model.eval()
    total, correct, n = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        p = model(x)
        total += loss_fn(p, y).item() * y.size(0)
        pred = (p >= 0.5).float()
        correct += (pred == y).sum().item()
        n += y.size(0)
    return total / max(n, 1), correct / max(n, 1)


for epoch in range(1, EPOCHS + 1):
    model.train()
    run_loss = 0.0
    run_correct = 0
    run_n = 0
    for x, y in train_dl:
        x, y = x.to(DEVICE), y.to(DEVICE)
        opt.zero_grad(set_to_none=True)
        p = model(x)
        loss = loss_fn(p, y)
        loss.backward()
        opt.step()
        run_loss += loss.item() * y.size(0)
        run_correct += ((p >= 0.5).float() == y).sum().item()
        run_n += y.size(0)

    tr_loss = run_loss / max(run_n, 1)
    tr_acc = run_correct / max(run_n, 1)
    va_loss, va_acc = evaluate(val_dl)
    print(
        f"epoch {epoch:02d}/{EPOCHS}  train loss {tr_loss:.4f} acc {tr_acc:.3f}  "
        f"val loss {va_loss:.4f} acc {va_acc:.3f}"
    )

tr_final_loss, tr_final_acc = evaluate(train_dl)
print(
    f"Train (eval mode, same split): loss {tr_final_loss:.4f} acc {tr_final_acc:.4f}"
)

_det_tag = "_".join(DATA_VERS)
ckpt = _REPO / "checkpoints" / f"gcn_detector_{_det_tag}.pt"
ckpt.parent.mkdir(parents=True, exist_ok=True)
torch.save(
    {
        "model_state": model.state_dict(),
        "data_vers": DATA_VERS,
        "caches": [str(p) for p in _CACHES],
    },
    ckpt,
)
print(f"Checkpoint → {ckpt.relative_to(_REPO)}")


ConcatDataset: 1 files, 8000 windows total
  V7: 8000
OVERFIT_SANITY: weight_decay=0, backbone dropout=0 — expect train acc → ~1.0 (val may degrade).
epoch 01/80  train loss 0.5871 acc 0.672  val loss 0.6247 acc 0.674
epoch 02/80  train loss 0.3524 acc 0.852  val loss 0.2959 acc 0.873
epoch 03/80  train loss 0.3216 acc 0.864  val loss 0.3020 acc 0.863
epoch 04/80  train loss 0.3045 acc 0.873  val loss 0.2928 acc 0.878
epoch 05/80  train loss 0.2851 acc 0.881  val loss 0.2949 acc 0.863
epoch 06/80  train loss 0.2866 acc 0.884  val loss 0.2769 acc 0.874
epoch 07/80  train loss 0.2774 acc 0.885  val loss 0.3107 acc 0.854


KeyboardInterrupt: 

In [5]:
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(trainable_params)

4725537


## Next steps:
Before all that, check to see if data extracted and used is good by overlaying.
1. Train model fully
2. Pick frames out of excel. And clips outside of excel. Run mdoel on them averaging output for frames making predictions
3. See model performance on that.


## Punch-type classification (`GCNClassifier`)

Trains **separately** from the punch/no-punch detector. Uses **`Dataset/landmarks.npz`**: each row is one annotated clip with a **six-class** punch label.

- **Windows**: `prepare_windows(..., window=CLF_WINDOW)` gives `(N, 10, 12, 2)` → batch shape **`[N, 1, 10, 12, 2]`**.
- **Labels**: strings in the npz (`Jab`, `Cross`, …) map to indices **`0 … 5`** in **`GCN.PUNCH_CLASSES`** order.

Generate **`landmarks.npz`** first (e.g. `python preprocess.py extract` from repo root).

In [ ]:
# Canonical pipeline (variable-length clips, xlsx alignment): run from repo root
%run train_classifier.py
